<a href="https://colab.research.google.com/github/ahmedchohan412/SMIT/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Smart Bus Ticket Booking & Reservation System
# Python 3, built-in functionality only. Data is lost when the program closes.

routes = [
    {"route_id":"R101","departure_city":"Karachi","destination_city":"Hyderabad","departure_time":"08:00 AM","ticket_price":1200,"total_seats":40,"available_seats":40},
    {"route_id":"R102","departure_city":"Karachi","destination_city":"Sukkur","departure_time":"09:30 AM","ticket_price":2000,"total_seats":36,"available_seats":36},
    {"route_id":"R103","departure_city":"Hyderabad","destination_city":"Mirpur Khas","departure_time":"11:00 AM","ticket_price":800,"total_seats":32,"available_seats":32},
    {"route_id":"R104","departure_city":"Karachi","destination_city":"Lahore","departure_time":"07:00 PM","ticket_price":5000,"total_seats":45,"available_seats":45},
    {"route_id":"R105","departure_city":"Sukkur","destination_city":"Hyderabad","departure_time":"02:00 PM","ticket_price":1500,"total_seats":30,"available_seats":30}
]
supported_cities = ("Karachi", "Hyderabad", "Sukkur", "Mirpur Khas", "Lahore")
bookings = []
issued_booking_ids = set()
next_booking_number = 1001

print("=" * 50)
print("SMART BUS TICKET BOOKING SYSTEM")
print("=" * 50)

while True:
    print("\n1. View Available Routes")
    print("2. Search for a Route")
    print("3. Book a Ticket")
    print("4. View My Bookings")
    print("5. Cancel a Booking")
    print("6. View Seat Availability")
    print("7. Generate Booking Report")
    print("8.  Exit")

    choice = input("Enter your choice (1-8): ").strip()

    if not choice.isdigit():
        print("Please enter a number from 1 to 8.")
        continue
    choice = int(choice)

    if choice == 1:
        print("\nAVAILABLE BUS ROUTES")
        for number, route in enumerate(routes, start=1):
            if route["available_seats"] == 0:
                status = "FULL"
            else:
                status = str(route["available_seats"]) + "/" + str(route["total_seats"])
            print(number, route["route_id"], route["departure_city"], "->",
                  route["destination_city"], "|", route["departure_time"],
                  "| Rs.", route["ticket_price"], "| Seats:", status)

    elif choice == 2:
        term = input("Search by route ID or city: ").strip().lower()
        found = False
        for route in routes:
            if term == route["route_id"].lower():
                print(route["route_id"], route["departure_city"], "->",
                      route["destination_city"], "|", route["departure_time"],
                      "| Rs.", route["ticket_price"], "| Available:", route["available_seats"])
                found = True
                break
            elif term == route["departure_city"].lower() or term == route["destination_city"].lower():
                print(route["route_id"], route["departure_city"], "->",
                      route["destination_city"], "|", route["departure_time"],
                      "| Rs.", route["ticket_price"], "| Available:", route["available_seats"])
                found = True
        if not found:
            print("No route matched your search.")

    elif choice == 3:
        name = input("Passenger name: ").strip()
        if not name:
            print("Name cannot be empty.")
            continue
        age_text = input("Passenger age: ").strip()
        if not age_text.isdigit() or int(age_text) <= 0:
            print("Age must be a positive whole number.")
            continue
        age = int(age_text)
        route_id = input("Route ID (e.g. R101): ").strip().upper()
        selected_route = None
        for route in routes:
            if route["route_id"] == route_id:
                selected_route = route
                break
        if selected_route is None:
            print("Route ID not found. No booking created.")
            continue
        seats_text = input("Number of seats: ").strip()
        if not seats_text.isdigit() or int(seats_text) <= 0:
            print("Seats must be a positive whole number.")
            continue
        seats = int(seats_text)
        if seats > selected_route["available_seats"]:
            print("Not enough seats available:", selected_route["available_seats"])
            continue
        booking_id = "B" + str(next_booking_number)
        while booking_id in issued_booking_ids:
            next_booking_number += 1
            booking_id = "B" + str(next_booking_number)
        fare = selected_route["ticket_price"] * seats
        booking = {"booking_id": booking_id, "passenger_name": name, "passenger_age": age,
                   "route_id": route_id, "seats": seats, "total_fare": fare, "status": "Confirmed"}
        bookings.append(booking)
        issued_booking_ids.add(booking_id)
        selected_route["available_seats"] -= seats
        next_booking_number += 1
        print("\nBOOKING CONFIRMED!")
        print("Booking ID:", booking_id, "| Passenger:", name, "| Age:", age)
        print("Route:", route_id, "| Seats:", seats, "| Fare per seat: Rs.", selected_route["ticket_price"])
        print("Total fare: Rs.", fare, "| Status: Confirmed")

    elif choice == 4:
        if not bookings:
            print("No bookings found.")
            continue
        for number, booking in enumerate(bookings, start=1):
            print("\nBooking", number, "| ID:", booking["booking_id"])
            print("Passenger:", booking["passenger_name"], "| Age:", booking["passenger_age"])
            print("Route:", booking["route_id"], "| Seats:", booking["seats"])
            print("Fare: Rs.", booking["total_fare"], "| Status:", booking["status"])

    elif choice == 5:
        booking_id = input("Booking ID to cancel: ").strip().upper()
        found = False
        for booking in bookings:
            if booking["booking_id"] == booking_id:
                found = True
                if booking["status"] != "Confirmed":
                    print("This booking has already been cancelled.")
                    break
                for route in routes:
                    if route["route_id"] == booking["route_id"]:
                        route["available_seats"] += booking["seats"]
                        break
                booking["status"] = "Cancelled"
                print("Booking", booking_id, "cancelled. Seats restored:", booking["seats"])
                break
        else:
            print("Booking ID not found.")
        if not found:
            print("No changes were made.")

    elif choice == 6:
        filter_choice = input("Show (1) all routes or (2) routes with seats available? ").strip()
        if filter_choice not in ("1", "2"):
            print("Invalid filter; showing all routes.")
            filter_choice = "1"
        for route in routes:
            if filter_choice == "2" and route["available_seats"] == 0:
                continue
            if route["available_seats"] == route["total_seats"]:
                status = "Available"
            elif route["available_seats"] == 0:
                status = "Fully Booked"
            else:
                status = "Partially Booked"
            print(route["route_id"], route["departure_city"], "->", route["destination_city"],
                  "|", route["available_seats"], "/", route["total_seats"], "|", status)

    elif choice == 7:
        total = len(bookings)
        confirmed = 0
        cancelled = 0
        reserved_seats = 0
        revenue = 0
        full_routes = 0
        for booking in bookings:
            if booking["status"] == "Confirmed":
                confirmed += 1
            elif booking["status"] == "Cancelled":
                cancelled += 1
        # Nested loops count only confirmed seats and revenue.
        for route in routes:
            if route["available_seats"] == 0:
                full_routes += 1
            for booking in bookings:
                if booking["route_id"] == route["route_id"] and booking["status"] == "Confirmed":
                    reserved_seats += booking["seats"]
                    revenue += booking["total_fare"]
        print("\nBOOKING REPORT")
        print("Total bookings created:", total)
        print("Confirmed bookings:", confirmed)
        print("Cancelled bookings:", cancelled)
        print("Seats currently reserved:", reserved_seats)
        print("Confirmed revenue: Rs.", revenue)
        print("Fully booked routes:", full_routes)
        print("\nCONFIRMED BOOKINGS BY ROUTE")
        for route in routes:
            count = 0
            seats_count = 0
            for booking in bookings:
                if booking["route_id"] == route["route_id"] and booking["status"] == "Confirmed":
                    count += 1
                    seats_count += booking["seats"]
            print(route["route_id"], "| Bookings:", count, "| Seats:", seats_count)


    elif choice == 8:
        print("Thank you for using Smart Bus Reservation System. Goodbye!")
        break
    else:
        print("Unsupported choice. Select a number from 1 to 8.")


SMART BUS TICKET BOOKING SYSTEM

1. View Available Routes
2. Search for a Route
3. Book a Ticket
4. View My Bookings
5. Cancel a Booking
6. View Seat Availability
7. Generate Booking Report
8.  Exit
Enter your choice (1-8): 3
Passenger name: ahmed
Passenger age: 18
Route ID (e.g. R101): R106
Route ID not found. No booking created.

1. View Available Routes
2. Search for a Route
3. Book a Ticket
4. View My Bookings
5. Cancel a Booking
6. View Seat Availability
7. Generate Booking Report
8.  Exit
Enter your choice (1-8): 3
Passenger name: ali
Passenger age: 15
Route ID (e.g. R101): R101
Number of seats: 5

BOOKING CONFIRMED!
Booking ID: B1001 | Passenger: ali | Age: 15
Route: R101 | Seats: 5 | Fare per seat: Rs. 1200
Total fare: Rs. 6000 | Status: Confirmed

1. View Available Routes
2. Search for a Route
3. Book a Ticket
4. View My Bookings
5. Cancel a Booking
6. View Seat Availability
7. Generate Booking Report
8.  Exit
Enter your choice (1-8): 5
Booking ID to cancel: B101
Booking ID no